# 01 — Models, training, evaluation and export of the test logits

DeepLabV3+ with an EfficientNet-B5 encoder for thyroid nodule segmentation on TN3K.
Four variants (no attention, GCA, SE, CBAM) are trained with two losses (Dice-BCE, Boundary), for eight configurations in total.

**What this notebook does**
1. Setup: dependencies, paths (Google Colab or local machine), seeds.
2. Data: TN3K, train/validation split (85/15, seed 42), augmentation.
3. Models and losses.
4. Metrics and training loops.
5. Training of **one** configuration, chosen in Section 5 (`ATTENTION`, `LOSS`). Run the notebook once per configuration.
6. Test-set size strata (thresholds used by the detection rates).
7. Evaluation of the eight checkpoints on the TN3K test set (Tables 1 and 2 of the paper).
8. Export of the test masks (8A, CPU) and of the raw test logits of the eight models (8B): input of notebooks 02, 03 and 04.

The conformal analyses (notebooks 02 and 03) and the model comparison (notebook 04) start from the logits exported here and run on a CPU.

Hardware: a GPU is needed for Sections 5, 7 and 8 (we used a T4 on Google Colab). With the checkpoints of the GitHub release (see README), skip Section 5.

## 1. Setup

### 1A. Dependencies

Colab already provides `torch`, `opencv` and `tqdm`; we pin the segmentation library and Albumentations to the versions used for the paper.
On a local machine install `requirements-gpu.txt` instead.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q segmentation-models-pytorch==0.3.4 albumentations==1.4.0

### 1B. Imports, seeds and device

In [ ]:
import os
import math
import random
import shutil
import zipfile
from collections import Counter
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from scipy.ndimage import label
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

SEED = 42


def seed_everything(seed: int = SEED) -> None:
    """Seed Python, NumPy and PyTorch (CPU and GPU)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")

### 1C. Paths

All notebooks use the same project folder:

```
PROJECT_ROOT/
    data/Thyroid Dataset/tn3k/   TN3K images and masks
    checkpoints/                 trained weights
    cache/logits/                test logits, masks and file names
```

On Colab `PROJECT_ROOT` is `MyDrive/thyroid_seg` on Google Drive, so data and checkpoints survive session restarts.
On a local machine it is the root of this repository, unless the environment variable `PROJECT_ROOT` is set.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/thyroid_seg")
else:
    repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", repo_root))

DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"
CACHE_DIR = PROJECT_ROOT / "cache" / "logits"
for d in (DATA_DIR, CHECKPOINTS_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

## 2. Data

### 2A. Get TN3K

TN3K (Gong et al., 2021) is distributed by its authors; the download link is in the [TRFE-Net repository](https://github.com/haifangong/TRFE-Net-for-thyroid-nodule-segmentation). We do not redistribute the images.

One-time manual step: download `Thyroid Dataset.zip` and put it in `PROJECT_ROOT/data/`. The cell below extracts it. Expected layout:

```
data/Thyroid Dataset/tn3k/
    trainval-image/   trainval-mask/   (2879 images)
    test-image/       test-mask/       (614 images)
```

In [ ]:
TN3K_ROOT = DATA_DIR / "Thyroid Dataset" / "tn3k"

if not TN3K_ROOT.exists():
    archive = DATA_DIR / "Thyroid Dataset.zip"
    if not archive.exists():
        raise FileNotFoundError(f"Put 'Thyroid Dataset.zip' (from the TRFE-Net repository) in {DATA_DIR}")
    print(f"Extracting {archive.name} ...")
    with zipfile.ZipFile(archive) as z:
        z.extractall(DATA_DIR)

for sub in ["trainval-image", "trainval-mask", "test-image", "test-mask"]:
    print(f"{sub:15s} {len(list((TN3K_ROOT / sub).iterdir()))} files")

On Colab, reading thousands of small files from Drive is slow, so we copy the dataset to the local disk of the session once.

In [ ]:
if IN_COLAB:
    DATA_ROOT = Path("/content/tn3k_local")
    if not DATA_ROOT.exists():
        print("Copying the dataset from Drive to the local disk ...")
        shutil.copytree(TN3K_ROOT, DATA_ROOT)
else:
    DATA_ROOT = TN3K_ROOT
print(f"Reading images from {DATA_ROOT}")

### 2B. PyTorch datasets

In [ ]:
class TN3KDataset(Dataset):
    """TN3K dataset with optional RAM preloading for faster training."""

    FLAT_MAP = {
        "trainval": ("trainval-image", "trainval-mask"),
        "train":    ("trainval-image", "trainval-mask"),
        "test":     ("test-image",     "test-mask"),
    }

    def __init__(self, root, split="trainval", transform=None, preload=True):
        self.root = Path(root)
        self.split = split
        self.transform = transform
        self.preload = preload

        if split in self.FLAT_MAP:
            img_folder, mask_folder = self.FLAT_MAP[split]
            flat_img  = self.root / img_folder
            flat_mask = self.root / mask_folder
            if flat_img.exists():
                self.image_dir = flat_img
                self.mask_dir  = flat_mask
            else:
                self.image_dir = self.root / split / "image"
                self.mask_dir  = self.root / split / "mask"
        else:
            self.image_dir = self.root / split / "image"
            self.mask_dir  = self.root / split / "mask"

        if not self.image_dir.exists():
            raise FileNotFoundError(f"Image dir not found: {self.image_dir}")
        if not self.mask_dir.exists():
            raise FileNotFoundError(f"Mask dir not found: {self.mask_dir}")

        valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}
        self.image_paths = sorted(
            p for p in self.image_dir.iterdir()
            if p.suffix.lower() in valid_exts
        )
        if not self.image_paths:
            raise RuntimeError(f"No images found in {self.image_dir}")

        # Pre-load everything into RAM
        if self.preload:
            print(f"Preloading {len(self.image_paths)} images into RAM...", end=" ")
            self._images = []
            self._masks  = []
            for p in self.image_paths:
                img  = cv2.imread(str(p), cv2.IMREAD_COLOR)
                img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                mask = cv2.imread(str(self._find_mask(p)), cv2.IMREAD_GRAYSCALE)
                mask = (mask > 127).astype(np.uint8)
                self._images.append(img)
                self._masks.append(mask)
            print("done.")

    def __len__(self):
        return len(self.image_paths)

    def _find_mask(self, img_path):
        candidate = self.mask_dir / img_path.name
        if candidate.exists():
            return candidate
        matches = list(self.mask_dir.glob(f"{img_path.stem}.*"))
        if not matches:
            raise FileNotFoundError(f"No mask for {img_path.name}")
        return matches[0]

    def __getitem__(self, idx):
        if self.preload:
            image = self._images[idx].copy()
            mask  = self._masks[idx].copy()
        else:
            image = cv2.imread(str(self.image_paths[idx]), cv2.IMREAD_COLOR)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mask  = cv2.imread(str(self._find_mask(self.image_paths[idx])), cv2.IMREAD_GRAYSCALE)
            mask  = (mask > 127).astype(np.uint8)

        if self.transform is not None:
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask  = transformed["mask"]

        if isinstance(mask, torch.Tensor) and mask.ndim == 2:
            mask = mask.unsqueeze(0).float()
        elif isinstance(mask, np.ndarray):
            mask = torch.from_numpy(mask).unsqueeze(0).float()

        return {"image": image, "mask": mask, "filename": self.image_paths[idx].name}

In [ ]:
class TransformedSubset(Dataset):
    """Subset wrapper that applies a transform on top of a base dataset.

    The base dataset must return raw (un-transformed) image and mask arrays.
    This lets several subsets share a single in-RAM copy of the data while
    using different transforms (training with augmentation, validation without).

    Args:
        base_dataset: A TN3KDataset instance built with transform=None.
        indices:      Iterable of integer indices into the base dataset.
        transform:    Albumentations Compose to apply at __getitem__ time.
    """

    def __init__(self, base_dataset, indices, transform):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # Resolve the global index in the base dataset
        global_idx = self.indices[idx]

        # Read the raw arrays directly, bypassing the base dataset's own transform
        if self.base_dataset.preload:
            image = self.base_dataset._images[global_idx].copy()
            mask  = self.base_dataset._masks[global_idx].copy()
        else:
            img_path = self.base_dataset.image_paths[global_idx]
            image = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mask  = cv2.imread(
                str(self.base_dataset._find_mask(img_path)),
                cv2.IMREAD_GRAYSCALE,
            )
            mask = (mask > 127).astype(np.uint8)

        # Apply the subset-specific transform
        transformed = self.transform(image=image, mask=mask)
        image = transformed["image"]
        mask  = transformed["mask"]

        # Ensure the mask has shape (1, H, W) and float dtype
        if isinstance(mask, torch.Tensor) and mask.ndim == 2:
            mask = mask.unsqueeze(0).float()
        elif isinstance(mask, np.ndarray):
            mask = torch.from_numpy(mask).unsqueeze(0).float()

        filename = self.base_dataset.image_paths[global_idx].name
        return {"image": image, "mask": mask, "filename": filename}

### 2C. Augmentation

Images are resized to 256 x 256 and normalized with the ImageNet statistics (the encoder is pre-trained on ImageNet).
Training augmentation: horizontal flip, small shift/scale/rotation, brightness/contrast jitter.

In [ ]:
IMG_SIZE = 256

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def get_train_transform(size: int = IMG_SIZE) -> A.Compose:
    """Training-time augmentations + normalization."""
    return A.Compose([
        A.Resize(size, size),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=0.1,
            rotate_limit=20,
            border_mode=cv2.BORDER_CONSTANT,
            p=0.5,
        ),
        A.RandomBrightnessContrast(
            brightness_limit=0.2,
            contrast_limit=0.2,
            p=0.5,
        ),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def get_eval_transform(size: int = IMG_SIZE) -> A.Compose:
    """Validation/test transform: resize + normalize, no augmentation."""
    return A.Compose([
        A.Resize(size, size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

### 2D. Train / validation / test

The original TN3K split is kept. The 2879 training images are split 85/15 into training and validation with a dedicated random generator (seed 42), so the split does not depend on other random calls in the notebook.
The validation set selects the best epoch; the test set is used only for the final evaluation and for the conformal analyses.

In [ ]:
# One in-RAM copy of the 2879 training images, shared by the training and validation subsets
base_trainval = TN3KDataset(DATA_ROOT, split="trainval", transform=None, preload=True)

N_TRAINVAL = len(base_trainval)
VAL_FRACTION = 0.15

split_rng = np.random.RandomState(SEED)
all_indices = np.arange(N_TRAINVAL)
split_rng.shuffle(all_indices)

n_val = int(round(N_TRAINVAL * VAL_FRACTION))
val_indices   = all_indices[:n_val]
train_indices = all_indices[n_val:]

train_ds = TransformedSubset(base_trainval, train_indices, get_train_transform())
val_ds   = TransformedSubset(base_trainval, val_indices,   get_eval_transform())
test_ds  = TN3KDataset(DATA_ROOT, split="test", transform=get_eval_transform(), preload=True)

print(f"Train: {len(train_ds)} images (with augmentation)")
print(f"Val:   {len(val_ds)} images")
print(f"Test:  {len(test_ds)} images")    # expected 2447 / 432 / 614

## 3. Models and losses

All models share the same DeepLabV3+ with an EfficientNet-B5 encoder pre-trained on ImageNet, and output one raw logit per pixel (the sigmoid is applied outside the model).

| Name | Where the attention acts |
|---|---|
| Vanilla | no attention |
| GCA | grouped channel attention on a skip connection (G=4), on the bottleneck (G=16) and on the decoder output (G=8) |
| SE | one squeeze-and-excitation block on the 256-channel decoder output |
| CBAM | one CBAM block (channel + spatial attention) on the 256-channel decoder output |

SE and CBAM are inserted at the same point, so that the comparison isolates the attention mechanism.

**Note on names.** Checkpoints are saved as `best_<loss class>_<model class>.pt` (for example `best_DiceBCELoss_CBAM_DeepLabV3Plus.pt`), so the class names below must not be changed, otherwise the released checkpoints are not found.

### 3A. Vanilla DeepLabV3+

In [ ]:
def build_baseline_model(
    encoder_name: str = "efficientnet-b5",
    encoder_weights: str = "imagenet",
) -> nn.Module:
    """Vanilla DeepLabV3+ baseline."""
    return smp.DeepLabV3Plus(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=1,         # binary: nodule vs background
        activation=None,   # raw logits; the sigmoid is applied outside the model
    )

### 3B. GCA-DeepLabV3+

Grouped channel attention, adapted from Kang et al. (2025). The input channels are split into G groups; for each group an independent two-layer MLP (implemented as grouped 1x1 convolutions) computes channel weights from spatially pooled features.

In [ ]:
class GroupedChannelAttention(nn.Module):
    """Grouped Channel Attention.

    Args:
        channels: number of input channels (must be divisible by groups)
        groups: number of channel groups (G)
        reduction: channel reduction ratio R in the bottleneck MLP
    """

    def __init__(self, channels: int, groups: int = 8, reduction: int = 4):
        super().__init__()
        if channels % groups != 0:
            raise ValueError(
                f"channels ({channels}) must be divisible by groups ({groups})"
            )

        # Bottleneck size must be a multiple of groups for the grouped 1x1 conv to work
        bottleneck = max(channels // reduction, groups)
        bottleneck = (bottleneck // groups) * groups

        self.groups = groups
        self.fc1 = nn.Conv2d(channels, bottleneck, kernel_size=1, groups=groups, bias=False)
        self.fc2 = nn.Conv2d(bottleneck, channels, kernel_size=1, groups=groups, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Spatial squeeze: (B, C, H, W) -> (B, C, 1, 1)
        z = F.adaptive_avg_pool2d(x, 1)
        # Per-group MLP: reduce, activate, expand, sigmoid
        z = self.relu(self.fc1(z))
        z = self.sigmoid(self.fc2(z))
        # Channel-wise re-scaling
        return x * z


class GCA_DeepLabV3Plus(smp.DeepLabV3Plus):
    """DeepLabV3+ with GCA on a skip connection, on the bottleneck and on the decoder output."""

    def __init__(self, encoder_name, encoder_weights, in_channels, classes):
        super().__init__(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=in_channels,
            classes=classes,
        )
        self.gca_bottleneck = GroupedChannelAttention(channels=512, groups=16, reduction=4)
        self.gca_skip = GroupedChannelAttention(channels=40, groups=4, reduction=4)
        self.gca_decoder = GroupedChannelAttention(channels=256, groups=8, reduction=4)

    def forward(self, x):
        enc_feature = self.encoder(x)
        enc_feature[2] = self.gca_skip(enc_feature[2])
        enc_feature[5] = self.gca_bottleneck(enc_feature[5])
        dec_feature = self.decoder(*enc_feature)
        dec_feature = self.gca_decoder(dec_feature)
        output = self.segmentation_head(dec_feature)
        return output


def gca_build_model():
    return GCA_DeepLabV3Plus(
        encoder_name="efficientnet-b5",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
    )

### 3C. SE-DeepLabV3+

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation block (Hu et al., 2018).

    Recalibrates channel-wise feature responses by modelling
    interdependencies between channels. Lightweight: only two FC layers
    with a bottleneck of size channels // reduction.

    Args:
        channels:  Number of input (and output) channels.
        reduction: Bottleneck ratio. Default 16 is standard in literature.
    """

    def __init__(self, channels, reduction=16):
        super().__init__()

        # Squeeze: global average pooling collapses H x W -> 1 x 1,
        # one descriptor per channel summarising its global spatial response.
        self.squeeze = nn.AdaptiveAvgPool2d(1)

        # Excitation: two FC layers learn non-linear channel dependencies.
        # The first compresses channels -> channels // reduction, the second expands back.
        # The sigmoid produces per-channel weights in (0, 1).
        self.excitation = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # x: (B, C, H, W) -> squeeze (B, C, 1, 1) -> flatten + excitation (B, C)
        weight = self.excitation(self.squeeze(x))
        # reshape for broadcasting: (B, C, 1, 1)
        weight = weight.unsqueeze(-1).unsqueeze(-1)

        # Channel-wise rescaling: same spatial map, reweighted channels
        return x * weight


class SE_DeepLabV3Plus(nn.Module):
    """DeepLabV3+ with a single SE block at the decoder output.

    The SE block is inserted after the decoder feature fusion and before
    the classification head: one attention module acting on the final
    256-channel representation before the pixel-wise decision.

    Architecture:
        EfficientNet-B5 encoder -> ASPP -> decoder -> SE(256) -> Conv1x1 -> logit
    """

    def __init__(self, encoder_name="efficientnet-b5",
                 encoder_weights="imagenet", num_classes=1):
        super().__init__()

        # Base DeepLabV3+, identical to the Vanilla baseline
        self.base = smp.DeepLabV3Plus(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=num_classes,
        )

        # SE block on the 256-channel decoder output
        self.se = SEBlock(channels=256, reduction=16)

        # Our own classification head, so that SE sits between decoder and classifier
        self.head = nn.Conv2d(256, num_classes, kernel_size=1)

        # Disable the original head; decoder features are extracted manually in forward()
        self.base.segmentation_head = nn.Identity()

    def forward(self, x):
        # Multi-scale encoder features
        features = self.base.encoder(x)

        # Decoder: ASPP + skip connection, output (B, 256, H/4, W/4)
        decoder_output = self.base.decoder(*features)

        # Upsample to the input resolution (smp does this inside its segmentation head)
        decoder_output = nn.functional.interpolate(
            decoder_output,
            size=x.shape[2:],
            mode="bilinear",
            align_corners=False,
        )

        # SE attention on the 256-channel full-resolution feature map
        decoder_output = self.se(decoder_output)

        # Classification head: 256 -> 1 channel (logit)
        return self.head(decoder_output)


def build_se_model():
    return SE_DeepLabV3Plus(
        encoder_name="efficientnet-b5",
        encoder_weights="imagenet",
        num_classes=1,
    )

### 3D. CBAM-DeepLabV3+

In [ ]:
class CBAM(nn.Module):
    """Convolutional Block Attention Module (Woo et al., ECCV 2018).

    Channel attention followed by spatial attention. The channel branch uses
    both average and max pooling (SE uses only average pooling): the mean
    response and the peak response of each channel.

    The spatial branch learns where to focus with a 7x7 convolution on a
    two-channel map built from the channel-wise average and max.

    Args:
        channels:        Number of input channels.
        reduction:       Bottleneck ratio for the channel attention FC layers.
        spatial_kernel:  Kernel size of the spatial attention conv (7 recommended).
    """

    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()

        # --- Channel attention ---
        # Both pooling paths share the same FC layers, as in the original paper.
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.channel_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
        )
        self.channel_sigmoid = nn.Sigmoid()

        # --- Spatial attention ---
        # Input: 2-channel map (avg and max across channels). Output: 1-channel weight map.
        assert spatial_kernel % 2 == 1, "spatial_kernel must be odd"
        padding = spatial_kernel // 2
        self.spatial_conv = nn.Conv2d(
            2, 1, kernel_size=spatial_kernel, padding=padding, bias=False
        )
        self.spatial_sigmoid = nn.Sigmoid()

    def forward(self, x):
        # --- Channel attention ---
        # avg and max pools: (B, C, 1, 1) -> flatten -> (B, C);
        # shared FC on both, element-wise sum, then sigmoid
        avg_out = self.channel_fc(self.avg_pool(x))
        max_out = self.channel_fc(self.max_pool(x))
        channel_weight = self.channel_sigmoid(avg_out + max_out)
        channel_weight = channel_weight.unsqueeze(-1).unsqueeze(-1)    # (B, C, 1, 1)
        x = x * channel_weight

        # --- Spatial attention ---
        avg_spatial = x.mean(dim=1, keepdim=True)                        # (B, 1, H, W)
        max_spatial = x.max(dim=1, keepdim=True).values                  # (B, 1, H, W)
        spatial_input = torch.cat([avg_spatial, max_spatial], dim=1)     # (B, 2, H, W)
        spatial_weight = self.spatial_sigmoid(self.spatial_conv(spatial_input))  # (B, 1, H, W)
        return x * spatial_weight


class CBAM_DeepLabV3Plus(nn.Module):
    """DeepLabV3+ with a single CBAM block at the decoder output.

    Same insertion point as SE_DeepLabV3Plus (after decoder fusion, before
    the classification head): same position, different attention mechanism.

    Architecture:
        EfficientNet-B5 encoder -> ASPP -> decoder -> CBAM(256) -> Conv1x1 -> logit
    """

    def __init__(self, encoder_name="efficientnet-b5",
                 encoder_weights="imagenet", num_classes=1):
        super().__init__()

        self.base = smp.DeepLabV3Plus(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=num_classes,
        )

        # CBAM on the 256-channel decoder output
        self.cbam = CBAM(channels=256, reduction=16, spatial_kernel=7)

        # Classification head (replaces the original one)
        self.head = nn.Conv2d(256, num_classes, kernel_size=1)

        # Disable the original head; decoder features are extracted manually in forward()
        self.base.segmentation_head = nn.Identity()

    def forward(self, x):
        features = self.base.encoder(x)
        decoder_output = self.base.decoder(*features)

        # Upsample to the input resolution before applying attention
        decoder_output = nn.functional.interpolate(
            decoder_output,
            size=x.shape[2:],
            mode="bilinear",
            align_corners=False,
        )

        # CBAM: channel attention, then spatial attention
        decoder_output = self.cbam(decoder_output)

        return self.head(decoder_output)


def build_cbam_model():
    return CBAM_DeepLabV3Plus(
        encoder_name="efficientnet-b5",
        encoder_weights="imagenet",
        num_classes=1,
    )

### 3E. Losses

- **Dice-BCE**: binary cross-entropy on the logits plus soft Dice loss, with weight 1 each.
- **Boundary**: Dice-BCE plus an edge term. The edges of the ground truth and of the predicted probabilities are extracted with a 3x3 Laplacian kernel, and the edge term is the BCE between the two edge maps. Its weight $\lambda$ starts at 0 and grows by 0.02 every 10 epochs, up to 0.1 (`update_lambda`).

In [ ]:
class DiceBCELoss(nn.Module):
    """Combined Dice + BCE loss."""

    def __init__(self, dice_weight: float = 1.0, bce_weight: float = 1.0, smooth: float = 1e-6):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        bce_loss = self.bce(logits, target)

        probs = torch.sigmoid(logits)
        intersection = (probs * target).sum(dim=(1, 2, 3))
        union = probs.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        dice_loss = 1.0 - dice.mean()

        return self.bce_weight * bce_loss + self.dice_weight * dice_loss


class Boundary_Loss(nn.Module):
    """Dice-BCE plus a BCE between the Laplacian edge maps of prediction and ground truth."""

    def __init__(self, lambda_boundary: float = 0):
        super().__init__()
        self.lambda_boundary = lambda_boundary
        self.kernel_boundary = nn.Conv2d(
            in_channels=1,
            out_channels=1,
            kernel_size=[3, 3],
            stride=1,
            padding=1,
            bias=False,
        )
        # Fixed (not trained) Laplacian kernel
        self.kernel_boundary.weight.data = torch.tensor([[0, -1, 0], [-1, 4, -1], [0, -1, 0]]).float().view(1, 1, 3, 3)
        self.kernel_boundary.weight.requires_grad = False

    def update_lambda(self, epoch):
        """Weight of the edge term: +0.02 every 10 epochs, at most 0.1."""
        self.lambda_boundary = min(0.02 * (epoch // 10), 0.1)
        print(f"Boundary loss lambda: {self.lambda_boundary}")

    def forward(self, logits: torch.Tensor, target: torch.Tensor):
        boundary = (self.kernel_boundary(target) - target).clamp(0, 1)

        pred = torch.sigmoid(logits)
        pred_boundary = (self.kernel_boundary(pred) - pred).clamp(0, 1)

        binary_bce = F.binary_cross_entropy(pred_boundary, boundary)

        bce_loss = F.binary_cross_entropy_with_logits(logits, target)
        intersection = (pred * target).sum(dim=(1, 2, 3))
        union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
        smooth = 1e-6
        dice = (2.0 * intersection + smooth) / (union + smooth)
        dice_loss = 1.0 - dice.mean()

        return bce_loss + dice_loss + self.lambda_boundary * binary_bce

### 3F. Registry of models and losses

The configuration is chosen by name (Section 5) instead of commenting and uncommenting lines.
`CONFIGS` lists the eight trained configurations with the checkpoint file of each one; Sections 7 and 8 loop over it.

In [ ]:
MODEL_BUILDERS = {
    "Vanilla": build_baseline_model,
    "GCA":     gca_build_model,
    "SE":      build_se_model,
    "CBAM":    build_cbam_model,
}
LOSSES = {
    "DiceBCE":  DiceBCELoss,
    "Boundary": Boundary_Loss,
}


def checkpoint_name(model: nn.Module, loss_fn: nn.Module) -> str:
    """File name of the best checkpoint, e.g. best_DiceBCELoss_CBAM_DeepLabV3Plus.pt."""
    return f"best_{type(loss_fn).__name__}_{type(model).__name__}.pt"


# (tag used for the logit files, attention, loss, checkpoint file)
CONFIGS = [
    ("Vanilla_DiceBCE",  "Vanilla", "DiceBCE",  "best_DiceBCELoss_DeepLabV3Plus.pt"),
    ("Vanilla_Boundary", "Vanilla", "Boundary", "best_Boundary_Loss_DeepLabV3Plus.pt"),
    ("GCA_DiceBCE",      "GCA",     "DiceBCE",  "best_DiceBCELoss_GCA_DeepLabV3Plus.pt"),
    ("GCA_Boundary",     "GCA",     "Boundary", "best_Boundary_Loss_GCA_DeepLabV3Plus.pt"),
    ("SE_DiceBCE",       "SE",      "DiceBCE",  "best_DiceBCELoss_SE_DeepLabV3Plus.pt"),
    ("SE_Boundary",      "SE",      "Boundary", "best_Boundary_Loss_SE_DeepLabV3Plus.pt"),
    ("CBAM_DiceBCE",     "CBAM",    "DiceBCE",  "best_DiceBCELoss_CBAM_DeepLabV3Plus.pt"),
    ("CBAM_Boundary",    "CBAM",    "Boundary", "best_Boundary_Loss_CBAM_DeepLabV3Plus.pt"),
]

Sanity check: every model maps a (2, 3, 256, 256) batch to (2, 1, 256, 256) logits, and the checkpoint names match `CONFIGS`.

In [ ]:
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
for tag, att, loss_name, ckpt_file in CONFIGS:
    if loss_name != "DiceBCE":
        continue                                   # one check per architecture is enough
    m = MODEL_BUILDERS[att]().to(DEVICE).eval()
    with torch.no_grad():
        out = m(dummy)
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    assert out.shape == (2, 1, IMG_SIZE, IMG_SIZE)
    assert checkpoint_name(m, LOSSES[loss_name]()) == ckpt_file
    print(f"{att:8s} output {tuple(out.shape)}  trainable params {n_params:,}")
    del m

## 4. Metrics and training loops

### 4A. Metrics

- `compute_metrics`: per-image Dice, IoU, accuracy, precision, recall and MCC, averaged over the batch (used to monitor training).
- `mcc_from_counts`: MCC from pooled confusion counts.
- `compute_nodules_detected_rate`: a nodule (connected component of the ground truth) is detected when at least 50% of its pixels are predicted. Nodules are binned by area (small / medium / large, thresholds from Section 6) and by multiplicity of the image (single / multiple).

In [ ]:
@torch.no_grad()
def compute_metrics(
    logits: torch.Tensor,
    target: torch.Tensor,
    threshold: float = 0.5,
) -> Dict[str, float]:
    """Per-image Dice, IoU, accuracy, precision, recall, MCC, averaged over the batch."""
    eps = 1e-7
    preds = (torch.sigmoid(logits) > threshold).float()

    tp = (preds * target).sum(dim=(1, 2, 3))
    fp = (preds * (1 - target)).sum(dim=(1, 2, 3))
    fn = ((1 - preds) * target).sum(dim=(1, 2, 3))
    tn = ((1 - preds) * (1 - target)).sum(dim=(1, 2, 3))

    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    accuracy = (tp + tn) / (tp + fp + fn + tn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    mcc = (tp * tn - (fp * fn) + eps) / ((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn) + eps).sqrt()

    return {
        "dice": dice.mean().item(),
        "iou": iou.mean().item(),
        "accuracy": accuracy.mean().item(),
        "precision": precision.mean().item(),
        "recall": recall.mean().item(),
        "mcc": mcc.mean().item(),
    }


def mcc_from_counts(tp: float, fp: float, fn: float, tn: float) -> float:
    """Matthews Correlation Coefficient from pooled pixel counts.

    Returns a value in [-1, +1], and 0.0 when the denominator is zero
    (degenerate cases, e.g. an all-background prediction).
    """
    numerator = tp * tn - fp * fn

    # Square root of each factor, then the product: multiplying four large
    # counts first (order 1e28) would lose precision.
    denominator = (
        math.sqrt(tp + fp)
        * math.sqrt(tp + fn)
        * math.sqrt(tn + fp)
        * math.sqrt(tn + fn)
    )

    if denominator == 0:
        return 0.0
    return numerator / denominator


def compute_nodules_detected_rate(
    pred: torch.Tensor,
    target: torch.Tensor,
    q33: float,
    q67: float,
    threshold: float = 0.5,
) -> dict:
    """Per-nodule detection, stratified by size and multiplicity.

    Args:
        pred:      raw logits, shape (1, H, W) or (H, W)
        target:    binary mask, shape (1, H, W) or (H, W)
        q33:       pixel-area threshold between small and medium (data-driven)
        q67:       pixel-area threshold between medium and large (data-driven)
        threshold: used both to binarize the prediction and as the minimum
                   covered fraction for a nodule to count as detected

    Returns:
        Dict with keys 'small', 'medium', 'large', 'single', 'multiple',
        each containing {'detected': int, 'total': int}.
    """
    result = {
        "small":    {"detected": 0, "total": 0},
        "medium":   {"detected": 0, "total": 0},
        "large":    {"detected": 0, "total": 0},
        "single":   {"detected": 0, "total": 0},
        "multiple": {"detected": 0, "total": 0},
    }

    pred_array   = (torch.sigmoid(pred) > threshold).float().squeeze().cpu().numpy()
    target_array = target.squeeze().cpu().numpy().astype(np.uint8)

    labeled_mask, num_nodules = label(target_array)
    if num_nodules == 0:
        return result

    # Multiplicity group of the image, applied to all its nodules
    group = "single" if num_nodules == 1 else "multiple"

    for k in range(1, num_nodules + 1):
        nodule_mask = (labeled_mask == k)
        area        = int(nodule_mask.sum())
        overlap     = pred_array[nodule_mask].sum() / area
        detected    = int(overlap >= threshold)

        if area < q33:
            size_bin = "small"
        elif area < q67:
            size_bin = "medium"
        else:
            size_bin = "large"

        result[size_bin]["total"]    += 1
        result[size_bin]["detected"] += detected
        result[group]["total"]       += 1
        result[group]["detected"]    += detected

    return result

### 4B. Training and evaluation loops

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    metrics_acc = {"dice": 0.0, "iou": 0.0}
    n_batches = 0

    pbar = tqdm(loader, desc="  train", leave=False, ncols=80)
    for batch in pbar:
        images = batch["image"].to(device, non_blocking=True)
        masks  = batch["mask"].to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss   = loss_fn(logits, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        m = compute_metrics(logits, masks)
        for k in metrics_acc:
            metrics_acc[k] += m[k]
        n_batches += 1

        pbar.set_postfix(loss=f"{loss.item():.4f}", dice=f"{m['dice']:.4f}")

    return {
        "loss": total_loss / n_batches,
        **{k: v / n_batches for k, v in metrics_acc.items()},
    }


@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    metrics_acc = {k: 0.0 for k in ["dice", "iou", "accuracy", "precision", "recall"]}
    n_batches = 0

    pbar = tqdm(loader, desc="  eval ", leave=False, ncols=80)
    for batch in pbar:
        images = batch["image"].to(device, non_blocking=True)
        masks  = batch["mask"].to(device, non_blocking=True)

        logits = model(images)
        loss   = loss_fn(logits, masks)
        total_loss += loss.item()

        m = compute_metrics(logits, masks)
        for k in metrics_acc:
            metrics_acc[k] += m[k]
        n_batches += 1

        pbar.set_postfix(loss=f"{loss.item():.4f}", dice=f"{m['dice']:.4f}")

    return {
        "loss": total_loss / n_batches,
        **{k: v / n_batches for k, v in metrics_acc.items()},
    }

### 4C. Hyperparameters and data loaders

In [ ]:
BATCH_SIZE = 4          # fits EfficientNet-B5 at 256 x 256 on a T4; lower it if you run out of memory
NUM_EPOCHS = 100
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 30           # early stopping: epochs without improvement of the validation Dice
NUM_WORKERS = 2

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

## 5. Training run

Choose the configuration, then run the two cells below. The checkpoint with the best validation Dice is saved in `CHECKPOINTS_DIR`, together with the per-epoch history.
Skip this section if you use the released checkpoints.

The optimizer is created **after** the model: it keeps references to the parameters of the model that exists when it is created, so it must be built on the model that is actually trained.

In [ ]:
ATTENTION = "CBAM"      # "Vanilla", "GCA", "SE", "CBAM"
LOSS = "DiceBCE"        # "DiceBCE", "Boundary"

seed_everything(SEED)
model = MODEL_BUILDERS[ATTENTION]().to(DEVICE)
loss_fn = LOSSES[LOSS]().to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

ckpt_path = CHECKPOINTS_DIR / checkpoint_name(model, loss_fn)
print(f"{ATTENTION} + {LOSS} -> {ckpt_path.name}")

In [ ]:
best_dice = 0.0
epochs_no_improve = 0
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"Starting epoch {epoch}...", flush=True)
    if hasattr(loss_fn, "update_lambda"):          # only the Boundary loss has a schedule
        loss_fn.update_lambda(epoch)
    train_metrics = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE)
    val_metrics   = evaluate(model, val_loader, loss_fn, DEVICE)
    scheduler.step()

    history.append({
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}":   v for k, v in val_metrics.items()},
    })

    print(
        f"Epoch {epoch:03d} | "
        f"train: loss {train_metrics['loss']:.4f}, dice {train_metrics['dice']:.4f} | "
        f"val: loss {val_metrics['loss']:.4f}, dice {val_metrics['dice']:.4f}, "
        f"iou {val_metrics['iou']:.4f}"
    )

    # Keep the checkpoint with the best validation Dice
    if val_metrics["dice"] > best_dice:
        best_dice = val_metrics["dice"]
        epochs_no_improve = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_dice": best_dice,
            "config": {
                "attention": ATTENTION,
                "loss": LOSS,
                "encoder": "efficientnet-b5",
                "img_size": IMG_SIZE,
                "batch_size": BATCH_SIZE,
                "lr": LR,
            },
        }, ckpt_path)
        print(f"  -> saved checkpoint to {ckpt_path.name}")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"  -> early stopping at epoch {epoch}")
            break

history_df = pd.DataFrame(history)
history_df.to_csv(CHECKPOINTS_DIR / f"history_{ATTENTION}_{LOSS}.csv", index=False)
print(f"\nBest validation Dice: {best_dice:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[1].plot(history_df["epoch"], history_df["train_dice"], label="train")
axes[1].plot(history_df["epoch"], history_df["val_dice"], label="validation")
axes[1].set_title("Dice")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.show()

## 6. Test-set size strata

The detection rates are stratified by nodule size. The thresholds `q33` and `q67` are the 33rd and 67th percentiles of the area of the largest nodule of each test image (in pixels, at 256 x 256).

In [ ]:
def nodule_pixel_count(mask):
    """Areas of the connected components of a binary mask.

    Args:
        mask: torch.Tensor of shape (1, H, W) / (H, W) or numpy array (H, W), values in {0, 1}.

    Returns:
        List of ints sorted in descending order, one per connected component
        (empty if the mask has no foreground pixel).
    """
    if isinstance(mask, torch.Tensor):
        mask_2d = mask.detach().squeeze().cpu().numpy().astype(np.uint8)
    else:
        mask_2d = mask.squeeze().astype(np.uint8)

    labeled, n_nodules = label(mask_2d)
    if n_nodules == 0:
        return []

    counts = [int((labeled == k).sum()) for k in range(1, n_nodules + 1)]
    counts.sort(reverse=True)
    return counts


def build_dataset_metadata(dataset):
    """Per-image metadata: index, filename, nodule areas, number of nodules, total and largest area."""
    metadata = []
    for idx in range(len(dataset)):
        sample = dataset[idx]
        counts = nodule_pixel_count(sample["mask"])
        metadata.append({
            "index": idx,
            "filename": sample["filename"],
            "nodule_pixels": counts,
            "n_nodules": len(counts),
            "total_pixels": sum(counts),
            "main_size": counts[0] if counts else 0,
        })
    return metadata

In [ ]:
test_metadata = build_dataset_metadata(test_ds)

print("Number of nodules per image:", Counter(e["n_nodules"] for e in test_metadata))

sizes = [e["main_size"] for e in test_metadata if e["main_size"] > 0]
q33, q67 = np.percentile(sizes, 33), np.percentile(sizes, 67)
print(f"Size thresholds: q33 = {q33:.0f} px, q67 = {q67:.0f} px")

## 7. Evaluation of the eight models (Tables 1 and 2)

One pass over the test set per model:
- **global** metrics from the confusion counts pooled over all test pixels (Dice_g, IoU_g, MCC_g, precision, recall);
- **per-image** Dice and MCC, summarized by the mean (mDice), the median (mDice_50) and the 10th percentile (mDice_10);
- detection rates by size and by multiplicity.

In [ ]:
def evaluation_checkpoint_full(model, test_loader, device, threshold: float = 0.5,
                               q33: float | None = None, q67: float | None = None):
    """Global (pooled) and per-image metrics in a single pass over the test set.

    Returns a dict with:
      - global_metrics: dataset-level scores (Dice_g, MCC_g, stratified detection rates)
      - per_image_df:   DataFrame with one row per image (Dice, MCC, gt_area_px)
      - summary:        mean / median / 10th percentile of the per-image values
    """
    assert q33 is not None and q67 is not None, "Pass q33 and q67 explicitly (Section 6)."
    model.eval()
    eps = 1e-7

    tp_total = fp_total = fn_total = tn_total = 0.0
    per_image_records = []
    dr_acc = {k: {"detected": 0, "total": 0} for k in ["small", "medium", "large", "single", "multiple"]}

    with torch.no_grad():
        for batch in test_loader:
            images    = batch["image"].to(device)
            masks     = batch["mask"].to(device)          # (B, 1, H, W)
            filenames = batch["filename"]

            logits = model(images)
            preds  = (torch.sigmoid(logits) > threshold).float()

            # Pooled counts over the whole batch
            tp_total += (preds * masks).sum().item()
            fp_total += (preds * (1 - masks)).sum().item()
            fn_total += ((1 - preds) * masks).sum().item()
            tn_total += ((1 - preds) * (1 - masks)).sum().item()

            # Per-image metrics and detection rates
            for i in range(images.shape[0]):
                p = preds[i]
                m = masks[i]

                tp_i = (p * m).sum().item()
                fp_i = (p * (1 - m)).sum().item()
                fn_i = ((1 - p) * m).sum().item()
                tn_i = ((1 - p) * (1 - m)).sum().item()

                per_image_records.append({
                    "filename":   filenames[i],
                    "dice":       (2 * tp_i + eps) / (2 * tp_i + fp_i + fn_i + eps),
                    "mcc":        mcc_from_counts(tp_i, fp_i, fn_i, tn_i),
                    "gt_area_px": m.sum().item(),
                })

                result = compute_nodules_detected_rate(logits[i], masks[i], q33=q33, q67=q67)
                for bin_name in dr_acc:
                    dr_acc[bin_name]["detected"] += result[bin_name]["detected"]
                    dr_acc[bin_name]["total"]    += result[bin_name]["total"]

    # --- Dataset-level (pooled) metrics
    dice_global = (2 * tp_total + eps) / (2 * tp_total + fp_total + fn_total + eps)
    iou_global  = (tp_total + eps) / (tp_total + fp_total + fn_total + eps)
    prec_global = (tp_total + eps) / (tp_total + fp_total + eps)
    rec_global  = (tp_total + eps) / (tp_total + fn_total + eps)
    mcc_global  = mcc_from_counts(tp_total, fp_total, fn_total, tn_total)

    def rate(k):
        return round(dr_acc[k]["detected"] / max(dr_acc[k]["total"], 1), 3)

    global_metrics = {
        "dice_global": round(dice_global * 100, 2),
        "iou_global":  round(iou_global  * 100, 2),
        "prec_global": round(prec_global * 100, 2),
        "rec_global":  round(rec_global  * 100, 2),
        "mcc_global":  round(mcc_global, 3),
        "DR_small":    rate("small"),
        "DR_medium":   rate("medium"),
        "DR_large":    rate("large"),
        "DR_single":   rate("single"),
        "DR_multiple": rate("multiple"),
        "n_images":    len(per_image_records),
    }

    # --- Per-image distribution
    per_image_df = pd.DataFrame(per_image_records)
    summary = {
        "mDice":    round(per_image_df["dice"].mean() * 100, 2),
        "mDice_50": round(per_image_df["dice"].median() * 100, 2),
        "mDice_10": round(per_image_df["dice"].quantile(0.10) * 100, 2),
        "mMCC":     round(per_image_df["mcc"].mean(), 3),
        "mMCC_50":  round(per_image_df["mcc"].median(), 3),
        "mMCC_10":  round(per_image_df["mcc"].quantile(0.10), 3),
    }
    return {"global_metrics": global_metrics, "summary": summary, "per_image_df": per_image_df}


def load_model(att: str, ckpt_file: str) -> nn.Module:
    """Build the architecture and load the trained weights."""
    model = MODEL_BUILDERS[att]().to(DEVICE)
    state = torch.load(CHECKPOINTS_DIR / ckpt_file, map_location=DEVICE)
    model.load_state_dict(state["model_state_dict"])
    return model.eval()

In [ ]:
rows = []
for tag, att, loss_name, ckpt_file in CONFIGS:
    if not (CHECKPOINTS_DIR / ckpt_file).exists():
        print(f"MISSING: {ckpt_file}")
        continue
    model = load_model(att, ckpt_file)
    out = evaluation_checkpoint_full(model, test_loader, DEVICE, q33=q33, q67=q67)
    rows.append({"model": f"{att} + {loss_name}", **out["global_metrics"], **out["summary"]})
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

results_df = pd.DataFrame(rows).set_index("model")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
print("Table 1: segmentation metrics")
print(results_df[["dice_global", "iou_global", "mcc_global", "mDice", "mDice_50", "mDice_10", "mMCC"]])
print("\nTable 2: detection rates")
print(results_df[["DR_small", "DR_medium", "DR_large", "DR_single", "DR_multiple"]])

## 8. Export for notebooks 02, 03 and 04

### 8A. Ground-truth masks and file names (CPU, no model needed)

The test masks are not redistributed with the release: they are derived from the TN3K annotations, so they are rebuilt here from the original images.
They come from the same test dataset used for evaluation (resize to 256 x 256, binarized at 127), in the order of the file names, which is also the order of the logits.

To run only the analyses (notebooks 02-04) with the released logits, run Sections 1 and 2, then this cell: no GPU and no checkpoint are needed.

In [ ]:
mask_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)
masks, names = [], []
for batch in mask_loader:
    masks.append(batch["mask"].squeeze(1))             # (B, 1, H, W) -> (B, H, W)
    names.extend(batch["filename"])
masks = (torch.cat(masks).numpy() > 0.5).astype(np.uint8)
np.save(CACHE_DIR / "test_masks.npy", masks)
(CACHE_DIR / "test_names.txt").write_text("\n".join(names))
print(masks.shape, "foreground fraction:", round(float(masks.mean()), 4))   # expected (614, 256, 256), about 0.12

### 8B. Test logits of the eight models (GPU)

For each model we save the raw logits on the 614 test images as `cache/logits/test_<tag>.npy` (float32, 614 x 256 x 256), from the same test dataset and in the same order as the masks of 8A.
The global Dice printed next to each model must match Table 1.

In [ ]:
@torch.no_grad()
def collect_logits(model, loader, device):
    """Raw logits (before the sigmoid) for the whole loader, shape (N, H, W)."""
    model.eval()
    out = []
    for batch in loader:
        logits = model(batch["image"].to(device))      # (B, 1, H, W)
        out.append(logits.squeeze(1).cpu())
    return torch.cat(out).numpy().astype(np.float32)


def dice_global(logits, masks):
    pred = logits > 0                                  # sigmoid(z) > 0.5  <=>  z > 0
    inter = (pred & (masks == 1)).sum()
    return 2 * inter / (pred.sum() + masks.sum())


masks = np.load(CACHE_DIR / "test_masks.npy")          # from 8A
for tag, att, loss_name, ckpt_file in CONFIGS:
    if not (CHECKPOINTS_DIR / ckpt_file).exists():
        print(f"MISSING: {ckpt_file}")
        continue
    model = load_model(att, ckpt_file)
    logits = collect_logits(model, test_loader, DEVICE)
    assert logits.shape == masks.shape, (logits.shape, masks.shape)
    np.save(CACHE_DIR / f"test_{tag}.npy", logits)
    print(f"{tag:18s} Dice_g = {100 * dice_global(logits, masks):.2f}")
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()